# B6 — User Profiles

Creates UserProfile per visitor with artwork-category preferences.
Since each visitor has exactly one trajectory in BIRD, EWMA reduces to a single observation.
Uses `profile_learning.py` with alpha=0.15.

In [ ]:
import os
import sys
import pandas as pd
from datetime import datetime

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint, update_endpoint
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX user:     <http://w3id.org/trajecto/user#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
"""

## 1. Get All Users and Their Category Interactions

In [ ]:
query = PREFIXES + """
SELECT ?user ?artworkType (SUM(?dwell) AS ?totalDwell) (COUNT(?interaction) AS ?visitCount)
WHERE {
    ?user a trajecto:Person .
    ?user trajecto:formsTrajectory ?traj .
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?interaction .
    ?interaction poi:interactsWith ?poi .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    ?interaction a poi:MeaningfulInteraction .
    ?poi poi:hasType ?artworkType .
}
GROUP BY ?user ?artworkType
"""
result = execute_sparql_query(query_endpoint, query)
df = format_results(result)
df['totalDwell'] = df['totalDwell'].astype(float)
df['visitCount'] = df['visitCount'].astype(int)
print(f'User-category pairs: {len(df)}')
print(f'Users: {df["user"].nunique()}')
df.head()

## 2. Compute Initial Preferences

For each user, normalize category interactions into preference scores [0, 1].

In [ ]:
from artwork_categories import resolve_category

# Map raw types to categories
df['category'] = df['artworkType'].apply(resolve_category)

# Aggregate by user + category
user_cats = df.groupby(['user', 'category']).agg(
    totalDwell=('totalDwell', 'sum'),
    visitCount=('visitCount', 'sum')
).reset_index()

# Compute preference score per user-category
# Score = weighted combination of visit proportion and dwell proportion
user_profiles = []
for user, group in user_cats.groupby('user'):
    total_dwell = group['totalDwell'].sum()
    total_visits = group['visitCount'].sum()

    for _, row in group.iterrows():
        dwell_ratio = row['totalDwell'] / total_dwell if total_dwell > 0 else 0
        visit_ratio = row['visitCount'] / total_visits if total_visits > 0 else 0
        pref_score = 0.7 * dwell_ratio + 0.3 * visit_ratio

        user_profiles.append({
            'user': user,
            'category': row['category'],
            'prefScore': round(pref_score, 4),
            'totalDwell': row['totalDwell'],
            'visitCount': row['visitCount'],
        })

profiles_df = pd.DataFrame(user_profiles)
print(f'User preference entries: {len(profiles_df)}')
profiles_df.head(10)

## 3. Insert User Profiles into KG

In [ ]:
now = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S')

insert_lines = []
profile_created = set()

for _, row in profiles_df.iterrows():
    user_uri = row['user']
    user_local = user_uri.split('#')[-1] if '#' in user_uri else user_uri.split('/')[-1]
    profile_uri = f'user:profile_{user_local}'
    pref_uri = f'user:pref_{user_local}_{row["category"]}'

    # Create profile once per user
    if user_uri not in profile_created:
        insert_lines.append(f'  {profile_uri} a user:UserProfile .')
        insert_lines.append(f'  {profile_uri} user:profileCreated "{now}"^^xsd:dateTime .')
        insert_lines.append(f'  <{user_uri}> user:hasProfile {profile_uri} .')
        profile_created.add(user_uri)

    # Category preference
    insert_lines.append(f'  {profile_uri} user:hasPreference {pref_uri} .')
    insert_lines.append(f'  {pref_uri} a user:PoiCategoryPreference .')
    insert_lines.append(f'  {pref_uri} user:forCategory "{row["category"]}" .')
    insert_lines.append(f'  {pref_uri} user:preferenceScore "{row["prefScore"]}"^^xsd:float .')

# Batch insert
batch_size = 500
total = 0
for i in range(0, len(insert_lines), batch_size):
    batch = insert_lines[i:i+batch_size]
    triples = '\n'.join(batch)
    update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
    if execute_sparql_update(update_endpoint, update):
        total += len(batch)

print(f'Inserted {total} user profile triples for {len(profile_created)} users')

## 4. Verify

In [ ]:
verification_queries = [
    ('UserProfiles', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a user:UserProfile }'),
    ('Preferences', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a user:PoiCategoryPreference }'),
    ('Users with profiles', 'SELECT (COUNT(DISTINCT ?u) AS ?c) WHERE { ?u user:hasProfile ?p }'),
]

for label, q in verification_queries:
    result = execute_sparql_query(query_endpoint, PREFIXES + q)
    df_v = format_results(result)
    count = df_v.iloc[0]['c'] if not df_v.empty else '?'
    print(f'{label}: {count}')

# Sample user profile
q = PREFIXES + """
SELECT ?user ?cat ?score
WHERE {
    ?user user:hasProfile ?profile .
    ?profile user:hasPreference ?pref .
    ?pref user:forCategory ?cat .
    ?pref user:preferenceScore ?score .
}
ORDER BY ?user DESC(?score)
LIMIT 20
"""
result = execute_sparql_query(query_endpoint, q)
format_results(result)